# Hyperparameter Tuning – FER-2013 ResNet18

This notebook performs a **grid search** over key hyperparameters (learning rate, batch size, augmentation strength, optimizer, scheduler) to find the best configuration for the FER-2013 emotion-classification task.

Every trained model is automatically saved with a descriptive filename following project nomenclature rules.

## Dataset Setup for Kaggle

#### FER2013
This notebook uses the FER-2013 dataset. To run on Kaggle:
1. Go to: https://www.kaggle.com/datasets/vikashkumar999/facial-expression-recognition
2. Click **"Add Data"** in the Kaggle notebook interface
3. Search for: `vikashkumar999/facial-expression-recognition`
4. The dataset will be available at `/kaggle/input/facial-expression-recognition/`

In [1]:
# ====== KAGGLE SETUP ======
# This notebook is designed to run on Kaggle.
# Make sure you've added the dataset:
# https://www.kaggle.com/datasets/vikashkumar999/facial-expression-recognition
# 
# In Kaggle notebook:
# 1. Click "Add Data" button
# 2. Search: vikashkumar999/facial-expression-recognition
# 3. Add the dataset to your notebook
# ==========================

In [2]:
# Quick environment check
import os
import sys

# Check if running on Kaggle
is_kaggle = os.path.exists('/kaggle')
print(f"Running on Kaggle: {is_kaggle}")
print(f"Python version: {sys.version.split()[0]}")

# Check GPU availability
try:
    import torch
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    else:
        print("⚠️ WARNING: No GPU detected! Enable GPU in Kaggle settings.")
except ImportError:
    print("PyTorch not yet imported")

Running on Kaggle: True
Python version: 3.12.12
PyTorch version: 2.8.0+cu126
CUDA available: True
GPU: Tesla P100-PCIE-16GB
GPU Memory: 17.06 GB


In [3]:
from pathlib import Path
import os

# ====== KAGGLE PATHS ======
# Dataset will be in Kaggle's input directory
DATASET_PATH = Path("/kaggle/input/facial-expression-recognition/FER_data.csv")

# Models will be saved to Kaggle's working directory (downloadable after run)
SAVE_DIR = Path("/kaggle/working/saved_models")

# ====== LOCAL PATHS (for testing locally, comment out Kaggle paths above) ======
# PROJECT_ROOT = Path.cwd().parents[1]
# DATASET_PATH = PROJECT_ROOT / "Datasets" / "FER_data.csv"
# SAVE_DIR = PROJECT_ROOT / "Models" / "Saved_Models"

# Ensure save directory exists
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Check dataset
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_PATH}\n"
        "Please add the dataset in Kaggle:\n"
        "1. Click 'Add Data' button\n"
        "2. Search: vikashkumar999/facial-expression-recognition\n"
        "3. Add to notebook"
    )
print(f"✓ Dataset found: {DATASET_PATH}")
print(f"✓ Models will be saved to: {SAVE_DIR}")

✓ Dataset found: /kaggle/input/facial-expression-recognition/FER_data.csv
✓ Models will be saved to: /kaggle/working/saved_models


In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import pandas as pd
import numpy as np
from PIL import Image
from collections import Counter
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score,
    accuracy_score,
)
import json, itertools, time, copy

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Torch: 2.8.0+cu126
CUDA available: True
GPU: Tesla P100-PCIE-16GB
Using device: cuda


## 1. Load & Inspect Dataset

In [5]:
data = pd.read_csv(DATASET_PATH)
data.columns = data.columns.str.strip()

print(f"Total samples : {len(data)}")
print(f"Columns       : {list(data.columns)}")
print(f"\nEmotion distribution:\n{data['emotion'].value_counts().sort_index()}")

FER_LABELS = {
    0: "angry",
    1: "disgust",
    2: "fear",
    3: "happy",
    4: "sad",
    5: "surprise",
    6: "neutral",
}
class_names = [FER_LABELS[i] for i in range(7)]
NUM_CLASSES = 7

Total samples : 35887
Columns       : ['emotion', 'Usage', 'pixels']

Emotion distribution:
emotion
0    4953
1     547
2    5121
3    8989
4    6077
5    4002
6    6198
Name: count, dtype: int64


## 2. Dataset Class & Augmentation Presets

In [6]:
class FERDataset(Dataset):
    """FER-2013 dataset: reads pixel strings, reshapes to 48x48 grayscale."""

    def __init__(self, dataframe, split, transform=None):
        self.data = dataframe[dataframe["Usage"] == split].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        label = int(row["emotion"])
        pixels = np.array(row["pixels"].split(), dtype=np.uint8).reshape(48, 48)
        image = Image.fromarray(pixels)
        if self.transform:
            image = self.transform(image)
        return image, label


# ---------- Augmentation presets ----------

def build_transform(aug_level: str = "medium"):
    """
    Returns a torchvision Compose pipeline.
    aug_level: 'light' | 'medium' | 'heavy'
    """
    base = [
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
    ]

    if aug_level == "light":
        aug = [
            transforms.RandomHorizontalFlip(p=0.3),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
        ]
    elif aug_level == "medium":
        aug = [
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(10),
            transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
        ]
    elif aug_level == "heavy":
        aug = [
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(15),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
            transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
        ]
    else:
        raise ValueError(f"Unknown aug_level: {aug_level}")

    tail = [
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]

    # RandomErasing must come AFTER ToTensor
    if aug_level == "heavy":
        erasing = aug.pop(-1)  # remove RandomErasing from aug
        return transforms.Compose(base + aug + tail + [erasing])

    return transforms.Compose(base + aug + tail)


# Validation transform (no augmentation)
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

print("Dataset class & augmentation presets ready.")

Dataset class & augmentation presets ready.


## 3. Compute Class Weights

In [7]:
train_labels = data[data["Usage"] == "Training"]["emotion"].values
counts = Counter(train_labels)
class_weights = torch.tensor(
    [1.0 / counts[i] for i in range(NUM_CLASSES)], dtype=torch.float
).to(device)

print("Class weights (inverse frequency):")
for i, w in enumerate(class_weights):
    print(f"  {FER_LABELS[i]:>10s}: {w:.6f}")

Class weights (inverse frequency):
       angry: 0.000250
     disgust: 0.002294
        fear: 0.000244
       happy: 0.000139
         sad: 0.000207
    surprise: 0.000315
     neutral: 0.000201


## 4. Hyperparameter Search Space

We tune across the following axes:
| Hyperparameter | Values |
|---|---|
| Learning Rate (fine-tune) | 5e-5, 1e-4, 3e-4 |
| Batch Size | 32, 64, 128 |
| Augmentation | light, medium, heavy |
| Optimizer | Adam, AdamW, SGD+momentum |
| LR Scheduler | StepLR, CosineAnnealing, ReduceLROnPlateau |

In [8]:
# ===================== SEARCH SPACE =====================

HYPERPARAM_GRID = {
    "lr": [5e-5, 1e-4, 3e-4],
    "batch_size": [32, 64, 128],
    "aug_level": ["light", "medium", "heavy"],
    "optimizer": ["adam", "adamw", "sgd"],
    "scheduler": ["step", "cosine", "plateau"],
}

# Phase-1: train classifier head only  |  Phase-2: fine-tune layer4 + fc
PHASE1_EPOCHS = 5
PHASE2_EPOCHS = 10
PATIENCE = 3  # early-stopping patience (phase 2)

# Generate all combinations
all_combos = list(itertools.product(
    HYPERPARAM_GRID["lr"],
    HYPERPARAM_GRID["batch_size"],
    HYPERPARAM_GRID["aug_level"],
    HYPERPARAM_GRID["optimizer"],
    HYPERPARAM_GRID["scheduler"],
))
print(f"Total hyperparameter combinations: {len(all_combos)}")
print("(You can slice `all_combos` below to run a subset.)")

Total hyperparameter combinations: 243
(You can slice `all_combos` below to run a subset.)


## 5. Training & Evaluation Helpers

In [9]:
def make_optimizer(name: str, params, lr: float):
    """Create optimizer by name."""
    if name == "adam":
        return torch.optim.Adam(params, lr=lr)
    elif name == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    elif name == "sgd":
        return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=1e-4)
    else:
        raise ValueError(f"Unknown optimizer: {name}")


def make_scheduler(name: str, optimizer, total_epochs: int):
    """Create LR scheduler by name."""
    if name == "step":
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
    elif name == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs)
    elif name == "plateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.5, patience=2
        )
    else:
        raise ValueError(f"Unknown scheduler: {name}")


def train_one_epoch(model, loader, optimizer, criterion):
    """Run one training epoch; return (avg_loss, accuracy)."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, desc="  Train", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)

        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{correct/total:.4f}")

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    """Evaluate model on a DataLoader; return accuracy."""
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return correct / total


@torch.no_grad()
def full_evaluation(model, loader):
    """
    Run full evaluation and return (y_true, y_pred, accuracy).
    """
    model.eval()
    all_preds, all_targets = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(1)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(labels.cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_targets)
    acc = accuracy_score(y_true, y_pred)
    return y_true, y_pred, acc


def build_model_name(combo_idx, lr, bs, aug, opt, sched, val_acc):
    """
    Build a filename following lowercase_underscore nomenclature.
    Example: resnet18_combo01_lr1e-04_bs64_aug_medium_opt_adam_sched_cosine_acc0.6312.pth
    """
    lr_str = f"{lr:.0e}".replace("+", "").replace("-0", "-")  # e.g. '1e-04'
    acc_str = f"{val_acc:.4f}"
    name = (
        f"resnet18_combo{combo_idx:02d}"
        f"_lr{lr_str}"
        f"_bs{bs}"
        f"_aug_{aug}"
        f"_opt_{opt}"
        f"_sched_{sched}"
        f"_acc{acc_str}"
        ".pth"
    )
    return name


print("Helper functions defined.")

Helper functions defined.


## 6. Hyperparameter Search Loop

Each configuration goes through:
1. **Phase 1** – Train only the classifier head (`fc` layer) for a few epochs.
2. **Phase 2** – Unfreeze `layer4` and fine-tune with a lower LR, using early stopping.

After Phase 2, the **best checkpoint** (by val accuracy) is saved with a descriptive filename.

### 🔄 Automatic Resume & Batch Training

The notebook now processes **ALL 243 combinations automatically** with smart resume:

- ✅ **No manual changes needed**: Just run the cell - it handles everything
- ✅ **Auto-skip trained models**: Detects existing models by matching hyperparameters  
- ✅ **Resume after interruption**: Kaggle timeout? Just re-run - it continues where it stopped
- ✅ **Run in batches**: Run for 2 hours, stop, come back later and run again
- ✅ **Zero duplicate work**: Already trained combinations are instantly skipped

**How it works:**
- Before training each combination, checks if a model file exists with those exact hyperparameters
- If found: loads the saved accuracy, adds to results, moves to next combination
- If not found: trains the model normally and saves it

**For testing:** Uncomment the line in the cell below to limit to a subset (e.g., 10 combinations)

### ⚠️ Important for Kaggle Users
- **GPU Required**: Enable GPU in Kaggle settings (Settings → Accelerator → GPU T4 x2)
- **Estimated time**: ~8 min per combo × number of untrained combos = total time
- **Full grid**: 243 combinations ≈ 30-35 hours total (spread across multiple runs)

In [10]:
# ===================== CHECK EXISTING MODELS =====================
# Check what models have already been trained

existing_model_files = sorted(SAVE_DIR.glob("resnet18_*.pth"))

print(f"📁 Found {len(existing_model_files)} existing model(s) in {SAVE_DIR}")

if existing_model_files:
    print("\n💾 Existing models (will be auto-skipped):")
    for model_file in existing_model_files[:5]:  # Show first 5
        try:
            ckpt = torch.load(model_file, map_location='cpu')
            acc = ckpt.get('best_val_acc', 'N/A')
            hp = ckpt.get('hyperparams', {})
            print(f"  • {model_file.name[:60]}...")
            print(f"    → Acc: {acc:.4f}, LR: {hp.get('lr', 'N/A')}, BS: {hp.get('batch_size', 'N/A')}, "
                  f"Aug: {hp.get('aug_level', 'N/A')}, Opt: {hp.get('optimizer', 'N/A')}")
        except:
            print(f"  • {model_file.name} (unable to read details)")
    
    if len(existing_model_files) > 5:
        print(f"  ... and {len(existing_model_files) - 5} more")
    
    print(f"\n✅ All {len(existing_model_files)} existing models will be auto-skipped!")
    print("   → The training loop below will only train new combinations")
else:
    print("  No existing models found - all combinations will be trained from scratch.")

📁 Found 0 existing model(s) in /kaggle/working/saved_models
  No existing models found - all combinations will be trained from scratch.


In [ ]:
# ===================== MAIN HYPER-TUNING LOOP =====================

# ⚠️ AUTOMATIC RESUME MODE ⚠️
# This processes ALL 243 combinations automatically.
# Already-trained combinations are skipped - just re-run this cell anytime!
# 
# To test with a subset first, uncomment this line:
# all_combos = all_combos[:10]  # Test with only 10 combinations

combos_to_run = all_combos  # Process all combinations - auto-skips trained ones!

print(f"🚀 Total hyperparameter combinations: {len(combos_to_run)}")
print(f"   Checking for existing models to skip...")
print()

results_log = []  # stores summary for every run
skipped_count = 0
trained_count = 0

# Validation dataset (same for all combinations)
val_dataset = FERDataset(data, split="PublicTest", transform=val_transform)

for combo_idx, (lr, bs, aug, opt_name, sched_name) in enumerate(combos_to_run, start=1):
    # ========== CHECK IF MODEL ALREADY EXISTS ==========
    # Search for existing model files with matching hyperparameters
    lr_str = f"{lr:.0e}".replace("+", "").replace("-0", "-")
    existing_models = list(SAVE_DIR.glob(f"resnet18_*_lr{lr_str}_bs{bs}_aug_{aug}_opt_{opt_name}_sched_{sched_name}_acc*.pth"))
    
    if existing_models:
        # Model already exists - load and skip training
        existing_path = existing_models[0]
        
        try:
            checkpoint = torch.load(existing_path, map_location=device)
            saved_acc = checkpoint.get("best_val_acc", 0.0)
            
            results_log.append({
                "combo_idx": combo_idx,
                "lr": lr,
                "batch_size": bs,
                "aug_level": aug,
                "optimizer": opt_name,
                "scheduler": sched_name,
                "best_val_acc": saved_acc,
                "model_file": existing_path.name,
            })
            
            skipped_count += 1
            
            # Show progress every 10 skips
            if skipped_count % 10 == 0 or skipped_count == 1:
                print(f"⏭️  Skipped {skipped_count} already-trained combinations...")
            
            continue  # Skip to next combination
            
        except Exception as e:
            print(f"\n⚠️  Error loading {existing_path.name}: {e}")
            print(f"→ Will re-train this combination\n")
    
    # ========== PROCEED WITH TRAINING (IF NOT SKIPPED) ==========
    
    tag = f"Combo {combo_idx}/{len(combos_to_run)} | lr={lr} bs={bs} aug={aug} opt={opt_name} sched={sched_name}"
    print(f"\n{'='*70}")
    print(f"  {tag}")
    print(f"  Progress: {skipped_count} skipped | {trained_count} trained | {combo_idx}/{len(combos_to_run)} total")
    print(f"{'='*70}")

    # ---- Data loaders ----
    train_transform = build_transform(aug)
    train_dataset = FERDataset(data, split="Training", transform=train_transform)

    train_loader = DataLoader(
        train_dataset, batch_size=bs, shuffle=True,
        num_workers=0, pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=bs, shuffle=False,
        num_workers=0, pin_memory=True,
    )

    # ---- Fresh model ----
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    # ==================== PHASE 1: Classifier Head ====================
    print("  Phase 1 — classifier head only")
    for param in model.parameters():
        param.requires_grad = False
    for param in model.fc.parameters():
        param.requires_grad = True

    optimizer = make_optimizer(opt_name, model.fc.parameters(), lr=lr * 10)  # higher LR for head
    scheduler = make_scheduler(sched_name, optimizer, PHASE1_EPOCHS)

    for epoch in range(1, PHASE1_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_acc = evaluate(model, val_loader)
        print(
            f"    [P1 Epoch {epoch}/{PHASE1_EPOCHS}] "
            f"Loss={train_loss:.4f}  TrainAcc={train_acc:.4f}  ValAcc={val_acc:.4f}"
        )
        if sched_name == "plateau":
            scheduler.step(val_acc)
        else:
            scheduler.step()

    # ==================== PHASE 2: Fine-tune layer4 + fc ====================
    print("  Phase 2 — fine-tune layer4 + fc")
    for param in model.layer4.parameters():
        param.requires_grad = True

    trainable_params = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = make_optimizer(opt_name, trainable_params, lr=lr)
    scheduler = make_scheduler(sched_name, optimizer, PHASE2_EPOCHS)

    best_val_acc = 0.0
    best_state = None
    patience_ctr = 0

    for epoch in range(1, PHASE2_EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_acc = evaluate(model, val_loader)
        print(
            f"    [P2 Epoch {epoch}/{PHASE2_EPOCHS}] "
            f"Loss={train_loss:.4f}  TrainAcc={train_acc:.4f}  ValAcc={val_acc:.4f}"
        )

        if sched_name == "plateau":
            scheduler.step(val_acc)
        else:
            scheduler.step()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"    Early stopping at epoch {epoch}.")
                break

    # ---- Restore best weights & save ----
    if best_state is not None:
        model.load_state_dict(best_state)
    model_filename = build_model_name(combo_idx, lr, bs, aug, opt_name, sched_name, best_val_acc)
    save_path = SAVE_DIR / model_filename

    torch.save({
        "model_state_dict": model.state_dict(),
        "num_classes": NUM_CLASSES,
        "hyperparams": {
            "lr": lr,
            "batch_size": bs,
            "aug_level": aug,
            "optimizer": opt_name,
            "scheduler": sched_name,
        },
        "best_val_acc": best_val_acc,
    }, save_path)
    trained_count += 1
    print(f"  ✓ Saved → {save_path.name}  (ValAcc={best_val_acc:.4f})")

    results_log.append({
        "combo_idx": combo_idx,
        "lr": lr,
        "batch_size": bs,
        "aug_level": aug,
        "optimizer": opt_name,
        "scheduler": sched_name,
        "best_val_acc": best_val_acc,
        "model_file": model_filename,
    })

    # Free GPU memory before next run
    del model, optimizer, scheduler, best_state
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\n{'='*70}")
print(f"✅ HYPERPARAMETER SEARCH COMPLETE")
print(f"{'='*70}")
print(f"Total combinations processed: {len(results_log)}")
print(f"⏭️  Skipped (already trained): {skipped_count}")
print(f"🆕 Newly trained this run: {trained_count}")
print(f"\n💾 All results logged and ready for analysis!")

🚀 Total hyperparameter combinations: 243
   Checking for existing models to skip...


  Combo 1/243 | lr=5e-05 bs=32 aug=light opt=adam sched=step
  Progress: 0 skipped | 0 trained | 1/243 total
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 200MB/s]


  Phase 1 — classifier head only


    [P1 Epoch 1/5] Loss=1.7611  TrainAcc=0.3272  ValAcc=0.3937


    [P1 Epoch 2/5] Loss=1.6236  TrainAcc=0.3894  ValAcc=0.3906


...

    [P2 Epoch 7/10] Loss=1.0296  TrainAcc=0.6061  ValAcc=0.6152


    [P2 Epoch 8/10] Loss=1.0021  TrainAcc=0.6140  ValAcc=0.6230


    [P2 Epoch 9/10] Loss=0.9808  TrainAcc=0.6201  ValAcc=0.6255


    [P2 Epoch 10/10] Loss=0.9814  TrainAcc=0.6223  ValAcc=0.6269
  ✓ Saved → resnet18_combo23_lr5e-5_bs32_aug_heavy_opt_adamw_sched_cosine_acc0.6269.pth  (ValAcc=0.6269)

  Combo 24/243 | lr=5e-05 bs=32 aug=heavy opt=adamw sched=plateau
  Progress: 0 skipped | 23 trained | 24/243 total
  Phase 1 — classifier head only


  Train:  28%|██▊       | 250/898 [00:35<01:28,  7.30it/s, acc=0.2124, loss=1.7166]

## 7. Results Summary & Best Model

In [ ]:
# ===================== RESULTS TABLE =====================
if not results_log:
    print("⚠️ No results to analyze. Results log is empty.")
    print("This might happen if:")
    print("  1. The training loop hasn't been run yet")
    print("  2. All cells above need to be executed first")
else:
    results_df = pd.DataFrame(results_log).sort_values("best_val_acc", ascending=False)
    results_df.to_csv(SAVE_DIR / "hypertuning_results.csv", index=False)

    print(results_df.to_string(index=False))
    print(f"\nResults saved to: {SAVE_DIR / 'hypertuning_results.csv'}")

    # ---- Identify best run ----
    best_run = results_df.iloc[0]
    print(f"\n{'='*60}")
    print(f"  BEST CONFIG  →  Combo {int(best_run['combo_idx'])}")
    print(f"  ValAcc       : {best_run['best_val_acc']:.4f}")
    print(f"  LR           : {best_run['lr']}")
    print(f"  Batch Size   : {int(best_run['batch_size'])}")
    print(f"  Augmentation : {best_run['aug_level']}")
    print(f"  Optimizer    : {best_run['optimizer']}")
    print(f"  Scheduler    : {best_run['scheduler']}")
    print(f"  Model File   : {best_run['model_file']}")
    print(f"{'='*60}")

## 8. Evaluate Best Model — Confusion Matrix & Metrics

In [ ]:
# ===================== LOAD & EVALUATE BEST MODEL =====================
if not results_log:
    print("⚠️ No results to evaluate. Run the training loop first (Cell 17).")
else:
    best_model_path = SAVE_DIR / best_run["model_file"]
    checkpoint = torch.load(best_model_path, map_location=device)

    best_model = models.resnet18(weights=None)
    best_model.fc = nn.Linear(best_model.fc.in_features, checkpoint["num_classes"])
    best_model.load_state_dict(checkpoint["model_state_dict"])
    best_model = best_model.to(device)
    best_model.eval()

    # ---- Validation loader (no augmentation) ----
    val_loader_eval = DataLoader(
        FERDataset(data, split="PublicTest", transform=val_transform),
        batch_size=64, shuffle=False, num_workers=0, pin_memory=True,
    )

    y_true, y_pred, acc = full_evaluation(best_model, val_loader_eval)
    print(f"Best model validation accuracy: {acc:.4f}")

    # ---- Confusion Matrix ----
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    plt.title(f"Confusion Matrix — Best Hypertuned Model (Acc={acc:.4f})")
    plt.colorbar()

    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45, ha="right")
    plt.yticks(tick_marks, class_names)

    thresh = cm.max() / 2
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(
                j, i, cm[i, j],
                horizontalalignment="center",
                color="white" if cm[i, j] > thresh else "black",
            )

    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.show()

    # ---- Classification Report ----
    print("\n===== Per-class Precision / Recall / F1 =====\n")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    # ---- F1 Scores ----
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")
    micro_f1 = f1_score(y_true, y_pred, average="micro")

    print("===== Overall F1 Scores =====")
    print(f"Macro F1    : {macro_f1:.4f}")
    print(f"Weighted F1 : {weighted_f1:.4f}")
    print(f"Micro F1    : {micro_f1:.4f}")

    # ---- Per-class Confusion Breakdown ----
    print("\n===== Per-class Confusion Matrix Breakdown =====")
    for i, cls in enumerate(class_names):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP
        TN = cm.sum() - (TP + FP + FN)
        print(f"\nClass: {cls}")
        print(f"  TP: {TP}  FP: {FP}  FN: {FN}  TN: {TN}")

## 9. Accuracy Comparison Bar Chart

In [ ]:
# ===================== VISUAL COMPARISON =====================
if not results_log:
    print("⚠️ No results to visualize. Run the training loop first (Cell 17).")
else:
    results_sorted = results_df.sort_values("best_val_acc", ascending=True)

    labels = [
        f"C{int(r['combo_idx'])} {r['optimizer']}|{r['scheduler']}\nlr={r['lr']} bs={int(r['batch_size'])} {r['aug_level']}"
        for _, r in results_sorted.iterrows()
    ]

    fig, ax = plt.subplots(figsize=(10, max(4, len(labels) * 0.45)))
    bars = ax.barh(range(len(labels)), results_sorted["best_val_acc"], color="steelblue")
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("Validation Accuracy")
    ax.set_title("Hyperparameter Tuning — All Runs Ranked")

    # Annotate bars
    for bar, val in zip(bars, results_sorted["best_val_acc"]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=8)

    plt.tight_layout()
    plt.show()

    print(f"\nAll {len(results_log)} models saved in: {SAVE_DIR}")


## 10. Download Results (Kaggle)

After the notebook finishes:
1. All models are saved in `/kaggle/working/saved_models/`
2. Results CSV: `/kaggle/working/saved_models/hypertuning_results.csv`
3. Click **"Save Version"** at top right
4. Once complete, go to **"Output"** tab to download all files
5. Or use the cell below to create a ZIP file

In [ ]:
# Optional: Create a ZIP file of all models for easy download
import shutil
from pathlib import Path

if Path("/kaggle").exists():  # Only on Kaggle
    zip_path = "/kaggle/working/all_hypertuned_models"
    
    print("Creating ZIP archive of all models...")
    shutil.make_archive(
        zip_path,
        'zip',
        SAVE_DIR
    )
    print(f"✓ ZIP created: {zip_path}.zip")
    print(f"  Size: {Path(f'{zip_path}.zip').stat().st_size / 1e6:.2f} MB")
    print("\nTo download:")
    print("1. Click 'Save Version' (top right)")
    print("2. Go to 'Output' tab")
    print("3. Download 'all_hypertuned_models.zip'")
else:
    print("Not running on Kaggle - skipping ZIP creation")